In [0]:
env = dbutils.widgets.get("environment") if dbutils.widgets.getAll().get("environment") else "dev"
bronzeTablName = f"saleslake_{env}.bronze_{env}.rawDiscount"
silverTablName = f"saleslake_{env}.silver_{env}.cleanedDiscount"

spark.sql(f"""
INSERT INTO {silverTablName}
SELECT DISTINCT
    CAST(TRIM(discount_id) AS INT)            AS discount_id,
    UPPER(TRIM(discount_code))                AS discount_code,
    TRIM(discount_name)                       AS discount_name,
    UPPER(TRIM(discount_type))                AS discount_type,
    CAST(TRIM(discount_value) AS DECIMAL(10,2)) AS discount_value,
    CAST(TRIM(min_amount) AS DECIMAL(18,2)) AS min_amount,
    CAST(TRIM(max_amount) AS DECIMAL(18,2)) AS max_amount,
    TO_DATE(TRIM(valid_from), 'yyyy-MM-dd')   AS valid_from,
    TO_DATE(TRIM(valid_to),   'yyyy-MM-dd')   AS valid_to,
    UPPER(TRIM(applicable_segment))           AS applicable_segment,
    UPPER(TRIM(applicable_category))          AS applicable_category,
    CAST(TRIM(usage_limit) AS INT) AS usage_limit,
    UPPER(TRIM(is_active))                    AS is_active,
    TO_DATE(TRIM(created_date), 'yyyy-MM-dd') AS created_date,
    CURRENT_TIMESTAMP() AS ingest_ts
FROM {bronzeTablName}
WHERE ingest_ts > (
    SELECT COALESCE(MAX(ingest_ts), TO_TIMESTAMP("1990-01-01"))
    FROM {silverTablName}
)
ORDER BY discount_id
""")

print(f"Silver load complete for {silverTablName}")
spark.sql(f"SELECT COUNT(*) AS row_count FROM {silverTablName}").display()